# 🚀 YOLOv5-C2f 口罩检测
> 自定义 C2f 模块 + Face Mask Detection 数据集 | T4 GPU

**Shift+Enter 逐个运行，不要一次性全部运行**

## 1. 检查 GPU + 挂载 Google Drive

In [ ]:
# 检查 GPU 是否可用（必须是 T4）
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 克隆仓库 + 安装依赖

In [ ]:
# 克隆你的仓库（包含 c2f 修改）
!git clone https://github.com/qianz7884-blip/yolo.git
%cd yolo

In [ ]:
# 安装依赖（requirements.txt 已修复）
!pip install -q -r requirements.txt
!pip install "Pillow<10.0.0"
!pip install -q kagglehub
print('✅ 依赖安装完成')

## 3. 下载数据集（Face Mask Detection）

In [ ]:
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path
import kagglehub

print('📥 下载 Face Mask Detection 数据集...')
src = Path(kagglehub.dataset_download('andrewmvd/face-mask-detection'))

# 创建 YOLO 目录结构（与 data/mask.yaml 的 path 对应）
dst = Path('./datasets/mask')
for split in ['train', 'val']:
    (dst / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dst / 'labels' / split).mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}

# 解析 XML 标注 → YOLO 格式
all_data = []
for xml_path in sorted((src / 'annotations').glob('*.xml')):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    size = root.find('size')
    w, h = int(size.find('width').text), int(size.find('height').text)

    objs = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in CLASS_MAP:
            continue
        bbox = obj.find('bndbox')
        xc = ((float(bbox.find('xmin').text) + float(bbox.find('xmax').text)) / 2) / w
        yc = ((float(bbox.find('ymin').text) + float(bbox.find('ymax').text)) / 2) / h
        bw = (float(bbox.find('xmax').text) - float(bbox.find('xmin').text)) / w
        bh = (float(bbox.find('ymax').text) - float(bbox.find('ymin').text)) / h
        objs.append(f'{CLASS_MAP[name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')

    if objs:  # 只保留有标注的图片
        all_data.append({'filename': filename, 'objects': objs})

# 80/20 分割（固定种子，可复现）
random.seed(42)
random.shuffle(all_data)
n = int(len(all_data) * 0.8)

for split, data in [('train', all_data[:n]), ('val', all_data[n:])]:
    for item in data:
        stem = Path(item['filename']).stem
        img_src = src / 'images' / item['filename']
        if img_src.exists():
            shutil.copy2(img_src, dst / 'images' / split / item['filename'])
        lbl_path = dst / 'labels' / split / f'{stem}.txt'
        lbl_path.write_text('\n'.join(item['objects']) + ('\n' if item['objects'] else ''))

print(f'✅ 训练集 {n} 张,  验证集 {len(all_data) - n} 张')

## 4. 训练（yolov5s-c2f）

In [ ]:
with open('/content/yolo/train.py', 'r') as f:
    text = f.read()

text = text.replace(
    "ckpt = torch.load(weights, map_location='cpu')",
    "ckpt = torch.load(weights, map_location='cpu', weights_only=False)"
)

with open('/content/yolo/train.py', 'w') as f:
    f.write(text)

print("修改完成")
!grep -n "torch.load(weights" /content/yolo/train.py
!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 50 \
    --data data/mask.yaml \
    --weights yolov5s.pt \
    --cfg models/yolov5s-c2f.yaml \
    --project /content/drive/MyDrive/yolov5-mask \
    --name exp-c2f \
    --workers 2

## 5. 查看训练结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib as mpl

# ==========================================
# 🌟 核心复刻代码：引入 Seaborn 样式 🌟
# ==========================================
try:
    import seaborn as sns
    sns.set_theme(style="darkgrid") # 完美的灰底白网格样式
except ImportError:
    print("正在安装 seaborn 库，请稍候...")
    !pip install seaborn -q
    import seaborn as sns
    sns.set_theme(style="darkgrid")

# 设置默认的中文字体和符号（防止乱码，如果不需要可以注释掉）
# mpl.rcParams['font.sans-serif'] = ['SimHei'] 
mpl.rcParams['axes.unicode_minus'] = False 

# ==========================================

# 1. 设置你的真实数据路径（保持不变）
data_dir = '/content/drive/MyDrive/yolov5-mask/exp-c2f'
csv_path = os.path.join(data_dir, 'results.csv')

# 2. 读取并清洗数据（保持不变）
if not os.path.exists(csv_path):
    print(f"❌ 错误：在 {data_dir} 中没有找到 results.csv，请确认路径是否正确。")
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip() # 去掉列名可能存在的空格
    
    # 3. 定义官方标准图的 10 个指标（保持不变）
    metrics = [
        'train/box_loss', 'train/obj_loss', 'train/cls_loss', 'metrics/precision', 'metrics/recall',
        'val/box_loss', 'val/obj_loss', 'val/cls_loss', 'metrics/mAP_0.5', 'metrics/mAP_0.5:0.95'
    ]
    
    # 兼容性检查：模糊匹配魔改版代码可能的列名改动（保持不变）
    for m in metrics:
        if m not in df.columns:
            matched = [col for col in df.columns if m in col]
            if matched:
                df.rename(columns={matched[0]: m}, inplace=True)

    # 4. 开始绘制标准的 2行5列 网格图（应用 Seaborn 样式）
    fig, axes = plt.subplots(2, 5, figsize=(18, 8), tight_layout=True)
    axes = axes.ravel() # 展平网格矩阵方便循环绘制
    
    for i, metric in enumerate(metrics):
        if metric in df.columns:
            # 完美的蓝色带点曲线 (results)
            # 调整了 markersize 以在 100 轮下依然清晰可视
            axes[i].plot(df[metric], marker='.', color='#1f77b4', label='results', markersize=3)
            
            # 红色虚线平滑曲线 (smooth) - 保持原图的虚线风格
            if len(df) > 5:
                smooth_val = df[metric].ewm(span=min(10, len(df))).mean()
                axes[i].plot(smooth_val, color='#d62728', linestyle=':', label='smooth', linewidth=1.5)
            
            axes[i].set_title(metric, fontsize=10)
            
            # 只在第二个子图（train/obj_loss）显示图例，保持原图风格
            if i == 1:
                axes[i].legend(loc='upper right', frameon=True)
        else:
            axes[i].text(0.5, 0.5, 'No Data', ha='center', va='center')
            axes[i].set_title(metric)

    # 5. 保存并展示结果
    # 重新命名为 results_replicated.png 以防覆盖之前的标准图
    save_path = os.path.join(data_dir, 'results.png')
    plt.savefig(save_path, dpi=300)
    plt.show()
    print(f"🎉 成功！你想要完美的“灰底白网格”复刻图已生成，并保存在：\n👉 {save_path}")




# from IPython.display import Image, display

# exp = '/content/drive/MyDrive/yolov5-mask/exp-c2f'


# print('📈 训练曲线')
# display(Image(filename=f'{exp}/results.png'))

# print('📊 混淆矩阵')
# display(Image(filename=f'{exp}/confusion_matrix.png'))

# print('🔍 验证样张')
# display(Image(filename=f'{exp}/val_batch0_pred.jpg'))

## 6. 推理检测（上传图片）

In [ ]:
from google.colab import files
import glob

uploaded = files.upload()
for fname in uploaded.keys():
    !python detect.py \
        --weights /content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt \
        --source {fname} \
        --conf 0.25
    result = glob.glob(f'/content/yolo/runs/detect/*/{fname}')
    if result:
        display(Image(filename=result[0]))

## 7. 下载模型

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/yolov5-mask/exp-c2f/weights/best.pt')